# 09 · Out-of-sample test of the simulator (2025)

Can the simulator predict races it hasn't seen? For each checkpoint (2025 rounds 7, 13 and 19):

1. **Refit** the driver/car model on everything before that round. The next block of races is added as forecast cells, so drivers and cars are carried forward from their latest data.
2. **Calibrate** overtaking thresholds on dry races before that round only.
3. **Simulate** each dry race in the block from its **real grid**, with the actual finishers, and rank drivers by mean simulated finish. Two versions:
   - **Simulator:** the model's forecast pace, made 1–6 races earlier.
   - **Simulator + weekend quali:** that forecast updated with the weekend's qualifying times (known before the race) by a Bayesian update: segment offsets estimated, ruined laps down-weighted, teammates' shared-car correlation kept.

This matches how the baseline model (notebook 06) was scored: grid known, finishing order among finishers predicted, same metrics, same drivers. Rounds 1–6 aren't tested, because the model starts each season's cars fresh and needs some 2025 data first.

**Runtime:** three fits of about 20–25 minutes each on Colab's free CPU. Each fit is saved to Drive as it finishes, so if Colab disconnects, rerun the notebook and finished checkpoints are loaded instead of refitted.

Run notebooks 00, 03, 05 and 07's install cell first (or run this notebook's install cell).

In [1]:
# @title Setup: mount Drive, get the latest code, install FastF1
REPO_URL = "https://github.com/khushi075/f1-race-sim.git"  # @param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/f1sim_data"  # @param {type:"string"}

import os, sys
from google.colab import drive

import os, sys
from google.colab import drive

try:
    import imp
except ImportError:
    import types, importlib
    imp = types.ModuleType('imp')
    imp.reload = importlib.reload
    sys.modules['imp'] = imp

drive.mount("/content/drive")

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "fastf1>=3.9,<4"

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["F1SIM_DATA_DIR"] = DATA_DIR

%load_ext autoreload
%autoreload 2
print("Code:", REPO_DIR, "| Data:", DATA_DIR)

Mounted at /content/drive
Cloning into '/content/f1-race-sim'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (146/146), done.
remote: Total 196 (delta 117), reused 110 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 165.28 KiB | 3.00 MiB/s, done.
Resolving deltas: 100% (117/117), done.
/content/f1-race-sim
Code: /content/f1-race-sim | Data: /content/drive/MyDrive/f1sim_data


In [2]:
# @title Install PyMC and the file format for saved fits (skipped if already installed)
import importlib.util
missing = [p for p in ("pymc", "h5netcdf", "h5py") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}

In [3]:
# @title Build the inputs
TEST_YEAR = 2025  # @param {type:"integer"}
CHECKPOINTS = "7, 13, 19"  # @param {type:"string"}
WET_RULE = "tyres"  # @param ["tyres", "tyres_or_rain"]

import pandas as pd
import plotly.express as px
from f1sim.paths import get_paths
from f1sim.ingest import load_quali
from f1sim.ingest_results import load_results
from f1sim.ingest_tracing import load_race_ti
from f1sim.race_pace import clean_dry_laps, driver_race_pace
from f1sim import backtest as bt, simulator as sim, skill_model as sm

paths = get_paths()
results = load_results(paths)
sim.calendar(results)  # fails with instructions if circuit info is missing
quali = load_quali(paths)
clean = clean_dry_laps(load_race_ti("laps", paths), wet_rule=WET_RULE)
race_pace, _ = driver_race_pace(clean)
dry_keys = set(map(tuple, clean[["Year", "Round"]].drop_duplicates().to_numpy()))
checkpoints = [int(c) for c in CHECKPOINTS.split(",")]
print("Blocks (first round, last round):",
      [(s, e - 1) for s, e in bt.blocks(results, TEST_YEAR, checkpoints)])

Blocks (first round, last round): [(7, 12), (13, 18), (19, 24)]


In [4]:
# @title Fit (or load) each checkpoint and simulate its races
DRAWS = 1000  # @param {type:"integer"}
TUNE = 1000  # @param {type:"integer"}
CHAINS = 2  # @param {type:"integer"}
REFIT = False  # @param {type:"boolean"}
N_SIMS = 3000  # @param {type:"integer"}

preds = []
for start, end in bt.blocks(results, TEST_YEAR, checkpoints):
    print(f"--- Checkpoint: data before round {start}; testing rounds {start}-{end - 1} ---")
    idata, data, how = bt.fit_checkpoint(quali, race_pace, results, TEST_YEAR, start, end, paths,
                                         draws=DRAWS, tune=TUNE, chains=CHAINS, refit=REFIT)
    diag = sm.diagnostics(idata)
    print(f"{how} | divergences {diag['divergences']} | max R-hat {diag['max_r_hat']:.3f} | "
          f"min bulk ESS {diag['min_ess_bulk']:.0f}")
    block_preds, _ = bt.score_block(sim.load_posterior(idata, data), results, dry_keys,
                                    TEST_YEAR, start, end, n_sims=N_SIMS,
                                    quali=quali)
    print(f"Simulated {block_preds.groupby('Round').ngroups} dry races")
    preds.append(block_preds)

sim_preds = pd.concat(preds, ignore_index=True)
sim_preds.to_parquet(paths.models_dir / f"backtest_{TEST_YEAR}_predictions.parquet", index=False)

--- Checkpoint: data before round 7; testing rounds 7-12 ---


Output()

fitted | divergences 0 | max R-hat 1.010 | min bulk ESS 402
Simulated 5 dry races
--- Checkpoint: data before round 13; testing rounds 13-18 ---


Output()

fitted | divergences 0 | max R-hat 1.010 | min bulk ESS 321
Simulated 5 dry races
--- Checkpoint: data before round 19; testing rounds 19-24 ---


Output()

fitted | divergences 0 | max R-hat 1.010 | min bulk ESS 383
Simulated 6 dry races


In [5]:
# @title Compare with grid order and the baseline models (same races, same drivers)
from f1sim.features import teammate_gaps
from f1sim.race_pace import teammate_race_gaps
from f1sim.baseline import build_dataset, fit_evaluate

base_df = build_dataset(results, teammate_gaps(quali), teammate_race_gaps(clean)[0])
baseline = fit_evaluate(base_df, test_year=TEST_YEAR)
print("Higher Spearman, lower MAE and higher Exact% are better.")
bt.compare(sim_preds, baseline["position_test_pred"])

Higher Spearman, lower MAE and higher Exact% are better.


,MeanSpearman,MAE_positions,ExactPct,Races,DriversScored
Grid order,0.763,2.362,25.177,16.0,282.0
Simulator,0.739,2.702,17.021,16.0,282.0
Simulator + weekend quali,0.792,2.305,24.113,16.0,282.0
Linear (ridge),0.805,2.305,20.567,16.0,282.0
Gradient boosting,0.798,2.312,19.858,16.0,282.0


In [6]:
# @title How well did the simulator's win probabilities pick the winners?
print(f"{sim_preds.groupby(['Year', 'Round']).ngroups} races. Higher win chance for the actual winner, "
      "more favourite wins and lower log-loss are better.")
bt.winner_scores(sim_preds)

16 races. Higher win chance for the actual winner, more favourite wins and lower log-loss are better.


,AvgWinChanceOfWinner%,FavouriteWon%,WinnerLogLoss
Simulator,28.340,50.00,1.339
Simulator + weekend quali,48.117,81.25,0.834


In [7]:
# @title Race by race: rank correlation with the real finishing order
from scipy.stats import spearmanr
rows = []
for (y, r), g in sim_preds.groupby(["Year", "Round"]):
    row = {"Round": r, "Event": g["Event"].iloc[0],
           "Grid order": spearmanr(g["FinishRank"], g["GridRank"]).statistic}
    for c in [c for c in g.columns if c.startswith("Pred (Simulator")]:
        row[c.removeprefix("Pred (").removesuffix(")")] = spearmanr(g["FinishRank"], g[c]).statistic
    rows.append(row)
per_race = pd.DataFrame(rows).melt(id_vars=["Round", "Event"], var_name="Model", value_name="Spearman")
px.line(per_race, x="Round", y="Spearman", color="Model", markers=True, hover_data=["Event"],
        title=f"{TEST_YEAR}: out-of-sample rank correlation by race").show()